In [35]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.manifold import TSNE
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import silhouette_score
from mpl_toolkits.mplot3d import Axes3D

In [36]:
#Part1: Data Preprocessing

#task_mode: 
#"early" removes potential leakage semester-performance features;
#"full" keeps all features.
task_mode = "early"
df_raw = pd.read_csv("../data/data.csv", sep=";")
df_raw.columns = df_raw.columns.str.strip()

#1.Missing values: add missing indicators + fill missing values
missing_cols = df_raw.columns[df_raw.isna().any()].tolist()
for col in missing_cols:
    df_raw[f"{col}_missing"] = df_raw[col].isna().astype(int)

num_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_raw.select_dtypes(exclude=[np.number]).columns.tolist()

for col in num_cols:
    if df_raw[col].isna().any():
        df_raw[col] = df_raw[col].fillna(df_raw[col].median())

for col in cat_cols:
    if df_raw[col].isna().any():
        mode_val = df_raw[col].mode(dropna=True)
        fill_val = mode_val.iloc[0] if not mode_val.empty else "Missing"
        df_raw[col] = df_raw[col].fillna(fill_val)

#2.Further processing
target_col = "Target"
X = df_raw.drop(columns=[target_col]).copy()
y = df_raw[target_col].copy()

#2.1 Leakage control for early-stage prediction
leakage_keywords = ["approved", "grade", "evaluations", "credited", "without evaluations"]
leakage_cols = [c for c in X.columns if any(k in c.lower() for k in leakage_keywords)]
if task_mode == "early":
    X = X.drop(columns=leakage_cols, errors="ignore")

#2.2 Treat coded categorical integers as categorical before one-hot
coded_categorical_cols = [
    "Marital status", "Application mode", "Application order", "Course",
    "Daytime/evening attendance", "Previous qualification", "Nacionality",
    "Mother's qualification", "Father's qualification",
    "Mother's occupation", "Father's occupation", "Displaced",
    "Educational special needs", "Debtor", "Tuition fees up to date",
    "Gender", "Scholarship holder", "International"
]
coded_categorical_cols = [c for c in coded_categorical_cols if c in X.columns]

for c in coded_categorical_cols:
    X[c] = X[c].astype("category")

#Keep boolean indicator for numeric binary columns not already converted
num_cols_X = X.select_dtypes(include=[np.number]).columns.tolist()
binary_numeric_cols = [
    c for c in num_cols_X if set(X[c].dropna().unique()).issubset({0, 1})
]
for c in binary_numeric_cols:
    X[c] = X[c].astype(bool)

#One-hot encode features and target (if target is non-numeric)
X_processed = pd.get_dummies(X, drop_first=False, dtype=int)
if not pd.api.types.is_numeric_dtype(y):
    y_processed = pd.get_dummies(y, prefix="Target", dtype=int)
else:
    y_processed = y

#2.3 Class-imbalance quick check
target_pct = y.value_counts(normalize=True) * 100

#2.4 Analyze if feature standardization is needed
num_cols_processed = X_processed.select_dtypes(include=[np.number]).columns.tolist()
binary_cols_processed = [
    c for c in num_cols_processed if set(X_processed[c].dropna().unique()).issubset({0, 1})
]
non_binary_cols_processed = [c for c in num_cols_processed if c not in binary_cols_processed]

if len(non_binary_cols_processed) > 1:
    stds = X_processed[non_binary_cols_processed].std(ddof=0)
    positive_stds = stds[stds > 0]
    if len(positive_stds) > 1:
        scale_ratio = positive_stds.max() / positive_stds.min()
    else:
        scale_ratio = 1.0
else:
    scale_ratio = 1.0

#Heuristic: if scale ratio is large, standardization is recommended
need_standardize = scale_ratio >= 10

X_standardized = X_processed.copy()
if non_binary_cols_processed:
    scaler = StandardScaler()
    X_standardized[non_binary_cols_processed] = scaler.fit_transform(
        X_standardized[non_binary_cols_processed]
    )

print(f"Raw dataset shape: {df_raw.shape}")
print(f"Columns with missing values: {len(missing_cols)}")
print(f"task_mode: {task_mode} | removed leakage cols: {len(leakage_cols) if task_mode == 'early' else 0}")
print(f"Coded categorical cols used: {len(coded_categorical_cols)}")
print(f"Features before encoding: {X.shape}, after encoding: {X_processed.shape}")
print("Target distribution (%):")
print(target_pct.round(2))

print("\nStandardization Analysis: ")
print(f"Non-binary numeric features: {len(non_binary_cols_processed)}")
print(f"Scale ratio (max_std/min_std among non-binary features): {scale_ratio:.2f}")
print(f"Need standardization (heuristic >= 10): {need_standardize}")

Raw dataset shape: (4424, 37)
Columns with missing values: 0
task_mode: early | removed leakage cols: 12
Coded categorical cols used: 18
Features before encoding: (4424, 24), after encoding: (4424, 250)
Target distribution (%):
Target
Graduate    49.93
Dropout     32.12
Enrolled    17.95
Name: proportion, dtype: float64

Standardization Analysis: 
Non-binary numeric features: 6
Scale ratio (max_std/min_std among non-binary features): 5.49
Need standardization (heuristic >= 10): False


### 6.1 Test Set spliting & Data Augmentation  

Following the usual procedure, the preprocessed data was divided into training set (80%) and test set (20%). 
For the training set, as seen in the preprocessing stage, the three target classes are severely imbalanced, which cause the model to neglect the minorities. To tackle this challenge, we introduced two strategies for data augmentation: the plain SMOTE algorithm and the Adaptive Synthetic (ADASYN) algorithm. The variables obtained from this stage is summarized as below:

Variable                    | Description
----------------------------|--------------------------------------------------
X_train, y_train            | Original training set (split from full data)
X_test, y_test              | Original test set (split from full data)
X_train_smote, y_train_smote | SMOTE-balanced training set
X_train_adasyn, y_train_adasyn | ADASYN-balanced training set

In [37]:
# Step 1: Prepare y as single column first
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE, ADASYN
import pandas as pd

# Prepare y as single column (not one-hot encoded)
if isinstance(y_processed, pd.DataFrame) and y_processed.shape[1] > 1:
    y_single = y_processed.idxmax(axis=1)
    y = y_single.str.replace('Target_', '')
else:
    y = y_processed

print("="*60)
print("ORIGINAL DATASET INFO:")
print("="*60)
print(f"Feature shape: {X_standardized.shape}")
print(f"Label shape: {len(y)}")
print("\nOriginal class distribution:")
for cls, count in y.value_counts().items():
    print(f"  {cls}: {count} samples ({count/len(y)*100:.2f}%)")
print("="*60)

# Step 2: Split into training and test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_standardized, 
    y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print("\n" + "="*60)
print("DATASET SPLIT COMPLETED:")
print("="*60)
print(f"Training set size: {len(X_train)} samples ({len(X_train)/len(y)*100:.1f}%)")
print(f"Test set size: {len(X_test)} samples ({len(X_test)/len(y)*100:.1f}%)")
print("\nTraining set original distribution:")
for cls, count in y_train.value_counts().items():
    print(f"  {cls}: {count} samples ({count/len(y_train)*100:.2f}%)")
print("\nTest set original distribution:")
for cls, count in y_test.value_counts().items():
    print(f"  {cls}: {count} samples ({count/len(y_test)*100:.2f}%)")
print("="*60)

# Step 3: Apply SMOTE to training set only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\n" + "="*60)
print("AFTER SMOTE ON TRAINING SET:")
print("="*60)
print(f"Training set size after SMOTE: {len(X_train_smote)} samples")
print(f"Synthetic samples added (SMOTE): {len(X_train_smote) - len(X_train)}")
print("\nSMOTE balanced distribution:")
for cls, count in y_train_smote.value_counts().items():
    print(f"  {cls}: {count} samples ({count/len(y_train_smote)*100:.2f}%)")
print("="*60)

# Step 4: Apply ADASYN to training set only (using same original training set)
adasyn = ADASYN(random_state=42)
X_train_adasyn, y_train_adasyn = adasyn.fit_resample(X_train, y_train)

print("\n" + "="*60)
print("AFTER ADASYN ON TRAINING SET:")
print("="*60)
print(f"Training set size after ADASYN: {len(X_train_adasyn)} samples")
print(f"Synthetic samples added (ADASYN): {len(X_train_adasyn) - len(X_train)}")
print("\nADASYN balanced distribution:")
for cls, count in y_train_adasyn.value_counts().items():
    print(f"  {cls}: {count} samples ({count/len(y_train_adasyn)*100:.2f}%)")
print("="*60)

# Step 5: Compare SMOTE vs ADASYN results
print("\n" + "="*60)
print("COMPARISON: SMOTE vs ADASYN")
print("="*60)

print("\nOriginal training set:")
for cls in sorted(y_train.unique()):
    orig_count = (y_train == cls).sum()
    print(f"  {cls}: {orig_count} samples")

print("\nAfter SMOTE:")
for cls in sorted(y_train_smote.unique()):
    smote_count = (y_train_smote == cls).sum()
    print(f"  {cls}: {smote_count} samples")

print("\nAfter ADASYN:")
for cls in sorted(y_train_adasyn.unique()):
    adasyn_count = (y_train_adasyn == cls).sum()
    print(f"  {cls}: {adasyn_count} samples")

print(f"\nTotal synthetic samples generated:")
print(f"  SMOTE: +{len(y_train_smote) - len(y_train)} samples")
print(f"  ADASYN: +{len(y_train_adasyn) - len(y_train)} samples")
print("="*60)

# Note: Test set remains unchanged for both methods
print("\n" + "="*60)
print("IMPORTANT NOTE:")
print("="*60)
print("Test set remains unchanged for both SMOTE and ADASYN:")
for cls, count in y_test.value_counts().items():
    print(f"  {cls}: {count} samples ({count/len(y_test)*100:.2f}%)")

ORIGINAL DATASET INFO:
Feature shape: (4424, 250)
Label shape: 4424

Original class distribution:
  Graduate: 2209 samples (49.93%)
  Dropout: 1421 samples (32.12%)
  Enrolled: 794 samples (17.95%)

DATASET SPLIT COMPLETED:
Training set size: 3539 samples (80.0%)
Test set size: 885 samples (20.0%)

Training set original distribution:
  Graduate: 1767 samples (49.93%)
  Dropout: 1137 samples (32.13%)
  Enrolled: 635 samples (17.94%)

Test set original distribution:
  Graduate: 442 samples (49.94%)
  Dropout: 284 samples (32.09%)
  Enrolled: 159 samples (17.97%)

AFTER SMOTE ON TRAINING SET:
Training set size after SMOTE: 5301 samples
Synthetic samples added (SMOTE): 1762

SMOTE balanced distribution:
  Graduate: 1767 samples (33.33%)
  Dropout: 1767 samples (33.33%)
  Enrolled: 1767 samples (33.33%)

AFTER ADASYN ON TRAINING SET:
Training set size after ADASYN: 5251 samples
Synthetic samples added (ADASYN): 1712

ADASYN balanced distribution:
  Dropout: 1808 samples (34.43%)
  Graduate:

### 6.2 Model Training

Regarding the algorithms used for building the classification models, we used Support Vector Machines (SVM), Random Forests (RF) and multi-layer perceptron (MLP). For each model, a 5-fold cross validation procedure was used to avoid overfitting. During the training process, F1 scores were computed for each class, and the average F1 score for the three classes was also computed. For hyperparameter optimization, we perform a grid search on SVM and random search for RF and MLP, using F1-score as the metric to be maximized. For the optimized model, accuracy was also computed as an overall metric.

#### 6.2.1 SVM

In [33]:
# One-vs-One SVM with hyperparameter tuning using fast grid (Using existing variables)
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Encode labels to numeric
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)
y_train_smote_encoded = le.transform(y_train_smote)
y_train_adasyn_encoded = le.transform(y_train_adasyn)
class_names = le.classes_

# Convert to numpy arrays
def to_numpy(data):
    return data.values if hasattr(data, 'values') else data

X_train_np = to_numpy(X_train)
X_train_smote_np = to_numpy(X_train_smote)
X_train_adasyn_np = to_numpy(X_train_adasyn)

# Define datasets - ALL use training data only
datasets = {
    'Original': (X_train_np, y_train_encoded),
    'SMOTE': (X_train_smote_np, y_train_smote_encoded),
    'ADASYN': (X_train_adasyn_np, y_train_adasyn_encoded)
}

# Reduced parameter grid for faster tuning
param_grid_fast = {
    'C': [0.1, 1, 10],
    'gamma': [0.1, 1, 'scale'],
    'kernel': ['rbf']
}

# Store results
results = {}
best_params_summary = {}

for dataset_name, (X_data, y_data) in datasets.items():
    print(f"\nProcessing {dataset_name} dataset...")
    print(f"  Samples: {len(X_data)}, Features: {X_data.shape[1]}")
    
    classes = np.unique(y_data)
    n_classes = len(classes)
    
    # Create class pairs
    class_pairs = [(classes[i], classes[j]) for i in range(n_classes) for j in range(i + 1, n_classes)]
    
    # Store metrics
    all_f1_scores = {cls: [] for cls in classes}
    all_accuracies = []
    fold_best_params = []
    
    # 5-fold CV with hyperparameter tuning
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_data, y_data), 1):
        X_train_fold = X_data[train_idx]
        X_val_fold = X_data[val_idx]
        y_train_fold = y_data[train_idx]
        y_val_fold = y_data[val_idx]
        
        # Train OVO models with GridSearchCV for each binary classifier
        ovo_models = {}
        
        for class_a, class_b in class_pairs:
            # Filter data for these two classes
            mask = (y_train_fold == class_a) | (y_train_fold == class_b)
            X_pair = X_train_fold[mask]
            y_pair = y_train_fold[mask]
            y_binary = (y_pair == class_b).astype(int)
            
            # Skip if not enough samples
            if len(np.unique(y_binary)) < 2:
                continue
            
            # Grid search for best parameters
            base_svm = SVC(random_state=42)
            grid_search = GridSearchCV(
                base_svm, 
                param_grid_fast, 
                cv=3, 
                scoring='f1',
                n_jobs=-1,
                verbose=0
            )
            grid_search.fit(X_pair, y_binary)
            
            # Use best estimator
            best_svm = grid_search.best_estimator_
            ovo_models[f"{class_a}_vs_{class_b}"] = best_svm
            fold_best_params.append(grid_search.best_params_)
        
        # Predict using voting
        n_val = X_val_fold.shape[0]
        votes = np.zeros((n_val, n_classes))
        
        for class_a, class_b in class_pairs:
            model_key = f"{class_a}_vs_{class_b}"
            if model_key in ovo_models:
                preds = ovo_models[model_key].predict(X_val_fold)
                for i, p in enumerate(preds):
                    if p == 0:
                        votes[i, class_a] += 1
                    else:
                        votes[i, class_b] += 1
        
        y_pred = classes[np.argmax(votes, axis=1)]
        
        # Calculate metrics
        acc = accuracy_score(y_val_fold, y_pred)
        all_accuracies.append(acc)
        
        f1_per_class = f1_score(y_val_fold, y_pred, labels=classes, average=None)
        for cls, f1 in zip(classes, f1_per_class):
            all_f1_scores[cls].append(f1)
    
    # Store results
    results[dataset_name] = {}
    for cls in classes:
        results[dataset_name][class_names[cls]] = np.mean(all_f1_scores[cls])
    results[dataset_name]['Macro Avg'] = np.mean([np.mean(all_f1_scores[cls]) for cls in classes])
    results[dataset_name]['Accuracy'] = np.mean(all_accuracies)
    
    # Store best params summary
    if fold_best_params:
        params_df = pd.DataFrame(fold_best_params)
        best_params_summary[dataset_name] = {}
        for param in ['C', 'gamma']:
            if param in params_df.columns:
                mode_val = params_df[param].mode()
                best_params_summary[dataset_name][param] = mode_val.iloc[0] if not mode_val.empty else 'N/A'

# Create results table
results_df = pd.DataFrame(results).T

# Reorder columns
desired_order = [c for c in class_names] + ['Macro Avg', 'Accuracy']
results_df = results_df[desired_order].round(4)

print("\n" + "="*80)
print("ONE-VS-ONE SVM F1-SCORE COMPARISON (WITH HYPERPARAMETER TUNING)")
print("="*80)
print("Note: All results are from 5-fold CV on training data only")
print("="*80)
print(results_df.to_string())
print("="*80)

# Display best parameters
print("\n" + "="*80)
print("BEST HYPERPARAMETERS (mode across folds):")
print("="*80)
for dataset_name, params in best_params_summary.items():
    print(f"\n{dataset_name}:")
    for param, value in params.items():
        print(f"  {param}: {value}")

# Save results
results_df.to_csv('ovo_svm_f1_comparison_tuned.csv')
print("\nResults saved to 'ovo_svm_f1_comparison_tuned.csv'")


Processing Original dataset...
  Samples: 3539, Features: 250

Processing SMOTE dataset...
  Samples: 5301, Features: 250

Processing ADASYN dataset...
  Samples: 5251, Features: 250

ONE-VS-ONE SVM F1-SCORE COMPARISON (WITH HYPERPARAMETER TUNING)
Note: All results are from 5-fold CV on training data only
          Dropout  Enrolled  Graduate  Macro Avg  Accuracy
Original   0.6290    0.1625    0.7560     0.5158     0.653
SMOTE      0.7477    0.7849    0.7603     0.7643     0.764
ADASYN     0.7180    0.7366    0.7572     0.7372     0.738

BEST HYPERPARAMETERS (mode across folds):

Original:
  C: 1
  gamma: scale

SMOTE:
  C: 1
  gamma: 0.1

ADASYN:
  C: 10
  gamma: 0.1

Results saved to 'ovo_svm_f1_comparison_tuned.csv'


From the above table, we can find that the SVM model without data over-sampling results in very low F1-score for the minority class (i.e. Enrolled), although producing a high accuracy. In contrast, the SMOTE and ADASYN methods are proved to significantly balance model's performance across all classes, as well as obtaining a higher accuracy. Morevoer, SMOTE outperforms ADASYN in all metircs.

#### 6.2.2 Random Forest

In [39]:
# Random Forest with 5-fold Cross Validation and Random Search hyperparameter tuning
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Encode labels
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_train_smote_encoded = le.transform(y_train_smote)
y_train_adasyn_encoded = le.transform(y_train_adasyn)
class_names = le.classes_

# Convert to numpy arrays
def to_numpy(data):
    return data.values if hasattr(data, 'values') else data

X_train_np = to_numpy(X_train)
X_train_smote_np = to_numpy(X_train_smote)
X_train_adasyn_np = to_numpy(X_train_adasyn)

# Define datasets - ALL use training data only
datasets = {
    'Original': (X_train_np, y_train_encoded),
    'SMOTE': (X_train_smote_np, y_train_smote_encoded),
    'ADASYN': (X_train_adasyn_np, y_train_adasyn_encoded)
}

# Random Forest parameter distribution for Random Search
param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [5, 10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'criterion': ['gini', 'entropy'],
    'class_weight': [None, 'balanced']
}

# Store results
results = {}
best_params_summary = {}
best_oob_scores = {}

for dataset_name, (X_data, y_data) in datasets.items():
    print(f"\nProcessing {dataset_name} dataset...")
    print(f"  Samples: {len(X_data)}, Features: {X_data.shape[1]}")
    
    # Create base Random Forest
    base_rf = RandomForestClassifier(random_state=42, n_jobs=-1)
    
    # Random Search with 5-fold cross validation
    random_search = RandomizedSearchCV(
        base_rf,
        param_distributions=param_dist,
        n_iter=30,
        cv=5,
        scoring='f1_macro',
        n_jobs=-1,
        random_state=42,
        verbose=0
    )
    
    random_search.fit(X_data, y_data)
    
    # Get best model and parameters
    best_rf = random_search.best_estimator_
    best_params_summary[dataset_name] = random_search.best_params_
    
    # Calculate OOB score for the optimized model (for reference only, not for evaluation)
    best_rf_with_oob = RandomForestClassifier(
        **random_search.best_params_,
        random_state=42,
        oob_score=True,
        n_jobs=-1
    )
    best_rf_with_oob.fit(X_data, y_data)
    best_oob_scores[dataset_name] = best_rf_with_oob.oob_score_
    
    # Perform 5-fold CV to get per-fold metrics
    all_f1_scores = {cls: [] for cls in np.unique(y_data)}
    all_accuracies = []
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    for train_idx, val_idx in cv.split(X_data, y_data):
        X_train_fold = X_data[train_idx]
        X_val_fold = X_data[val_idx]
        y_train_fold = y_data[train_idx]
        y_val_fold = y_data[val_idx]
        
        # Train with best parameters
        temp_rf = RandomForestClassifier(
            **random_search.best_params_,
            random_state=42,
            n_jobs=-1
        )
        temp_rf.fit(X_train_fold, y_train_fold)
        
        y_pred_fold = temp_rf.predict(X_val_fold)
        
        acc = accuracy_score(y_val_fold, y_pred_fold)
        all_accuracies.append(acc)
        
        f1_per_class = f1_score(y_val_fold, y_pred_fold, labels=np.unique(y_data), average=None)
        for cls, f1 in zip(np.unique(y_data), f1_per_class):
            all_f1_scores[cls].append(f1)
    
    # Store results
    results[dataset_name] = {}
    for cls in np.unique(y_data):
        results[dataset_name][class_names[cls]] = np.mean(all_f1_scores[cls])
    results[dataset_name]['Macro Avg'] = np.mean([np.mean(all_f1_scores[cls]) for cls in np.unique(y_data)])
    results[dataset_name]['Accuracy'] = np.mean(all_accuracies)

# Create results table
results_df = pd.DataFrame(results).T
desired_order = [c for c in class_names] + ['Macro Avg', 'Accuracy']
results_df = results_df[desired_order].round(4)

# Add OOB Score column (for reference only)
results_df['OOB Score'] = pd.Series(best_oob_scores).round(4)

# Reorder to put OOB Score at the end
desired_order_with_oob = [c for c in class_names] + ['Macro Avg', 'Accuracy', 'OOB Score']
results_df = results_df[desired_order_with_oob]

print("\n" + "="*85)
print("RANDOM FOREST F1-SCORE COMPARISON (RANDOM SEARCH + 5-FOLD CV)")
print("="*85)
print("Note: All metrics except OOB Score are from 5-fold CV on training data")
print("      OOB Score is from the optimized model on full training data (reference only)")
print("="*85)
print(results_df.to_string())
print("="*85)

# Display best parameters
print("\n" + "="*80)
print("BEST HYPERPARAMETERS (from Random Search):")
print("="*80)
for dataset_name, params in best_params_summary.items():
    print(f"\n{dataset_name}:")
    for param, value in params.items():
        print(f"  {param}: {value}")

# Save results
results_df.to_csv('rf_cv_comparison_with_oob.csv')
print("\nResults saved to 'rf_cv_comparison_with_oob.csv'")


Processing Original dataset...
  Samples: 3539, Features: 250

Processing SMOTE dataset...
  Samples: 5301, Features: 250

Processing ADASYN dataset...
  Samples: 5251, Features: 250

RANDOM FOREST F1-SCORE COMPARISON (RANDOM SEARCH + 5-FOLD CV)
Note: All metrics except OOB Score are from 5-fold CV on training data
      OOB Score is from the optimized model on full training data (reference only)
          Dropout  Enrolled  Graduate  Macro Avg  Accuracy  OOB Score
Original   0.6337    0.3544    0.7357     0.5746    0.6383     0.6349
SMOTE      0.7235    0.7764    0.7470     0.7490    0.7499     0.7546
ADASYN     0.7119    0.7373    0.7439     0.7311    0.7313     0.7429

BEST HYPERPARAMETERS (from Random Search):

Original:
  n_estimators: 50
  min_samples_split: 5
  min_samples_leaf: 2
  max_features: sqrt
  max_depth: 30
  criterion: gini
  class_weight: balanced

SMOTE:
  n_estimators: 200
  min_samples_split: 2
  min_samples_leaf: 1
  max_features: None
  max_depth: None
  criter

The adavantage of SMOTE and ADASYN over the original dataset is still distinctive, even from the perspective of OOB score, which is a special feature for Random Forest.

### 6.2.3 MLP

In [41]:
# MLP with 5-fold Cross Validation (Concise version)
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Encode labels
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_train_smote_encoded = le.transform(y_train_smote)
y_train_adasyn_encoded = le.transform(y_train_adasyn)
class_names = le.classes_

# Convert to numpy arrays
def to_numpy(data):
    return data.values if hasattr(data, 'values') else data

X_train_np = to_numpy(X_train)
X_train_smote_np = to_numpy(X_train_smote)
X_train_adasyn_np = to_numpy(X_train_adasyn)

# Define datasets
datasets = {
    'Original': (X_train_np, y_train_encoded),
    'SMOTE': (X_train_smote_np, y_train_smote_encoded),
    'ADASYN': (X_train_adasyn_np, y_train_adasyn_encoded)
}

# Fixed parameters (not tuned)
FIXED_PARAMS = {
    'random_state': 42,
    'max_iter': 500,
    'early_stopping': True,
    'validation_fraction': 0.1,
    'verbose': False
}

# Tuned parameters
param_dist = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate': ['adaptive'],
    'learning_rate_init': [0.0001, 0.001],
    'batch_size': [32, 64]
}

# Store results
results = {}
best_params_summary = {}

for dataset_name, (X_data, y_data) in datasets.items():
    print(f"\nProcessing {dataset_name} dataset...")
    print(f"  Samples: {len(X_data)}, Features: {X_data.shape[1]}")
    
    # Base MLP
    base_mlp = MLPClassifier(**FIXED_PARAMS)
    
    # Random Search
    random_search = RandomizedSearchCV(
        base_mlp, param_dist, n_iter=25, cv=5, 
        scoring='f1_macro', n_jobs=-1, random_state=42, verbose=0
    )
    random_search.fit(X_data, y_data)
    best_params_summary[dataset_name] = random_search.best_params_
    
    # 5-fold CV evaluation
    all_f1_scores = {cls: [] for cls in np.unique(y_data)}
    all_accuracies = []
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    for train_idx, val_idx in cv.split(X_data, y_data):
        # Merge fixed and best params
        mlp_params = {**FIXED_PARAMS, **random_search.best_params_}
        temp_mlp = MLPClassifier(**mlp_params)
        
        temp_mlp.fit(X_data[train_idx], y_data[train_idx])
        y_pred = temp_mlp.predict(X_data[val_idx])
        
        all_accuracies.append(accuracy_score(y_data[val_idx], y_pred))
        f1_per_class = f1_score(y_data[val_idx], y_pred, labels=np.unique(y_data), average=None)
        for cls, f1 in zip(np.unique(y_data), f1_per_class):
            all_f1_scores[cls].append(f1)
    
    # Store results
    results[dataset_name] = {}
    for cls in np.unique(y_data):
        results[dataset_name][class_names[cls]] = np.mean(all_f1_scores[cls])
    results[dataset_name]['Macro Avg'] = np.mean([np.mean(all_f1_scores[cls]) for cls in np.unique(y_data)])
    results[dataset_name]['Accuracy'] = np.mean(all_accuracies)

# Create results table
results_df = pd.DataFrame(results).T
desired_order = [c for c in class_names] + ['Macro Avg', 'Accuracy']
results_df = results_df[desired_order].round(4)

print("\n" + "="*80)
print("MLP F1-SCORE COMPARISON (RANDOM SEARCH + 5-FOLD CV)")
print("="*80)
print(results_df.to_string())
print("="*80)

print("\n" + "="*80)
print("BEST HYPERPARAMETERS (from Random Search):")
print("="*80)
for dataset_name, params in best_params_summary.items():
    print(f"\n{dataset_name}:")
    for param, value in params.items():
        print(f"  {param}: {value}")

results_df.to_csv('mlp_cv_comparison.csv')
print("\nResults saved to 'mlp_cv_comparison.csv'")


Processing Original dataset...
  Samples: 3539, Features: 250

Processing SMOTE dataset...
  Samples: 5301, Features: 250

Processing ADASYN dataset...
  Samples: 5251, Features: 250

MLP F1-SCORE COMPARISON (RANDOM SEARCH + 5-FOLD CV)
          Dropout  Enrolled  Graduate  Macro Avg  Accuracy
Original   0.6406    0.2819    0.7580     0.5602    0.6606
SMOTE      0.6822    0.7255    0.7240     0.7105    0.7119
ADASYN     0.6702    0.6718    0.7346     0.6922    0.6943

BEST HYPERPARAMETERS (from Random Search):

Original:
  solver: adam
  learning_rate_init: 0.001
  learning_rate: adaptive
  hidden_layer_sizes: (50,)
  batch_size: 32
  alpha: 0.01
  activation: relu

SMOTE:
  solver: adam
  learning_rate_init: 0.001
  learning_rate: adaptive
  hidden_layer_sizes: (100, 50)
  batch_size: 64
  alpha: 0.0001
  activation: relu

ADASYN:
  solver: adam
  learning_rate_init: 0.001
  learning_rate: adaptive
  hidden_layer_sizes: (100, 50)
  batch_size: 64
  alpha: 0.0001
  activation: relu

R

We can now conclued that of the three methods, SMOTE is the optimal one for data augmentation.

Moreover, it can be observed that SVM outperforms the other two models according to the F1-score and Accuracy.